## Spotify API Call using Spotipy

### Here I am importing the Spotipy package as well as setting the client ID and client Secret
### The client ID and secret are generated when you register an app on the Spotify Developer website

In [1]:
from spotipy.oauth2 import SpotifyClientCredentials #To access authorised Spotify data
from tqdm import tqdm_notebook
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import random
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
client_id = os.getenv("MY_CLIENT_ID")
client_secret = os.getenv("MY_API_KEY")
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager) #spotify object to access API


### This functiion calls the audio_features function which allows us to access the Spotify audio features endpoint
### The endpoint will allow a list of 100 tracks at a time which is what is being done here
### Care must be taken not to hit this endpoint more than 180 times per minute or the account will lock for 12 hours

In [2]:
def get_features(chunk):
 
    try:
        # Get audio features for the chunk of track IDs
        audio_features = list(sp.audio_features(chunk))
        #af.append(audio_features)

        time.sleep(30)

    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 429:
            retry_after = int(e.response.headers.get('Retry-After', 1))
            print(f"Rate limited. Retrying after {retry_after} seconds.")
            time.sleep(retry_after + 1)
        else:
            raise
    except Exception as e:
        print(f"Failed to get audio features for some track IDs: {e}")
        tempo_lst.extend([0.0] * len(chunk))

    return audio_features

### This function takes the list of tracks and their features and converts them to a dataframe

In [3]:
def df_tracks(tracklist):
    
    '''
    Takes the output of artist_tracks (i.e. a list of lists),
    puts it in a dataframe and formats it.
    '''

    df = pd.DataFrame(data=[tracklist], columns=['artist',
     'album_name',
     'album_uri',
     'track',
     'trackuri',
     'release_date',
     'danceability',
     'energy',
     'key',
     'loudness',
     'mode',
     'speechiness',
     'acousticness',
     'instrumentalness',
     'liveness',
     'valence',
     'tempo',
     'type',
     'id',
     'uri',
     'track_href',
     'analysis_url',
     'duration_ms',
     'time_signature'])

    df.rename(columns={'uri':'song_uri'}, inplace=True)

    df.drop_duplicates(subset=['artist', 'track', 'release_date'], inplace=True)

    # Reorder the cols to have identifiers first, auditory features last
    cols = ['artist', 'album_name', 'album_uri', 'track','trackuri', 'release_date', 'id', 'song_uri', 'track_href',
     'analysis_url', 'type', 'danceability', 'energy', 'key',  'loudness', 'mode', 'speechiness',
     'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']

    df = df[cols]
    
    return df

### This fetches a list of random playlists using the output of randint() as the seed

In [4]:
playlist_uris = []

for i in range(0,10):
    playlist_uris.append(sp.search(random.randint(0, 10000), type='playlist'))

### This calls the playlist_tracks funtion and passes in the playlist IDs retrieved in the cell above
### It returns a list of track information including artist,trackname, trackID etc..

In [6]:
playlist_tracks = []

for m in range(len(playlist_uris)):
    playlist_tracks.append(sp.playlist_tracks(playlist_uris[m]['playlists']['items'][0]['uri'], fields=None, limit=100, offset=0, market=None, additional_types=('track', )))

### Here I am extracting the required json values from the output generated in the cell above and putting in a list

In [7]:
trackList = []
trackLists = []

for q in range(len(playlist_tracks) - 1):
    for p in range(len(playlist_tracks[q]['items'])):
        trackList = [playlist_tracks[q]['items'][p]['track']['artists'][0]['name'],playlist_tracks[q]['items'][p]['track']['album']['name'],
            playlist_tracks[q]['items'][p]['track']['album']['uri'],playlist_tracks[q]['items'][p]['track']['name'],playlist_tracks[q]['items'][p]['track']['uri'], 
            playlist_tracks[q]['items'][p]['track']['album']['release_date']]
        trackLists.append(trackList)

### Here I am passing the lists of 100 trackIDs to the audio_features function to retrieve track freatures such as
### tempo, key, speechiness etc...

In [9]:
feats = []
chunk = []
chunks = []
final_tracks = []
f = 0
s = 0
    
for k in range(len(trackLists)):
    chunk.append(trackLists[k][4])
    if k%100 == 0:
        feat = get_features(chunk)
        chunk = []
        
        for j in range(f,k): 
            trackLists[j].extend(feat[s].values())
            s = s + 1
        final_tracks.append(trackLists)
        f = k
        s = 0

In [10]:
final_tracks[0][106]

['Taio Cruz',
 'Rokstarr',
 'spotify:album:159jpldDCgsgH6KGgjy63c',
 'Break Your Heart',
 'spotify:track:1CdqVF1ywD0ZO1zXtB9yWa',
 '2010-01-01',
 0.693,
 0.891,
 1,
 -3.261,
 1,
 0.296,
 0.0281,
 0,
 0.143,
 0.532,
 91.1,
 'audio_features',
 '4Q4K8fikwOmZYmz82Wco4w',
 'spotify:track:4Q4K8fikwOmZYmz82Wco4w',
 'https://api.spotify.com/v1/tracks/4Q4K8fikwOmZYmz82Wco4w',
 'https://api.spotify.com/v1/audio-analysis/4Q4K8fikwOmZYmz82Wco4w',
 268053,
 4]

### Here I am converting the list of tracks with thier features to a dataframe.

In [11]:

for p in range(len(final_tracks[0])):
    if (len(final_tracks[0][p]) == 24):
        if (p == 0):
            finaldf = df_tracks(final_tracks[0][p])
        else:
            tmpdf = df_tracks(final_tracks[0][p])
            finaldf = pd.concat([finaldf, tmpdf], axis=0)
    else: 
        break

In [13]:
len(finaldf)

700

In [14]:
finaldf = finaldf.drop(['type'], axis=1)

In [15]:
finaldf.to_csv('spotouttrack11.csv', index=False)